# IMF PortWatch: Ормузский пролив и Баб-эль-Мандеб

Этот ноутбук — аналог прежнего `suez_portwatch_downloader_excel_ready.ipynb`, но сразу для **двух проливов**:

- **Баб-эль-Мандеб:** `portid = 'chokepoint4'`
- **Ормузский пролив:** `portid = 'chokepoint6'`

Загружает **ежедневные** показатели PortWatch с 2019 года до даты запуска, отдельно `n_dry_bulk` (число сухогрузов) и `capacity_dry_bulk` (их оценочная грузовместимость), а также остальные типы судов. Выгружает CSV, Excel, Parquet, делает месячную таблицу для команды и строит простые графики.

**Смысл данных:** `n_*` — число проходов судов, `capacity_*` — оценочная грузовместимость, **не фактический объём экспорта карбамида**. Источник: [IMF PortWatch](https://portwatch.imf.org/).

## 0. Установка библиотек

Выполни эту ячейку первой. Если работаешь в Google Colab, установка выполняется в его окружении. Если в Jupyter — в окружении текущего ядра Python.

In [ ]:
%pip install -q requests pandas numpy matplotlib openpyxl pyarrow

## 1. Загрузка ежедневных данных PortWatch

Запросы к API выполняются **постранично**. Мы проверяем, что новые страницы не дублируют старые, и после загрузки удаляем возможные повторные строки для одного пролива и даты. Если PortWatch временно недоступен, появится понятная ошибка, а не фиктивный CSV.

`END_DATE` автоматически равна **сегодняшней дате** (можно заменить на определённую дату, если нужно воспроизвести выгрузку).

In [ ]:
from pathlib import Path
from datetime import date
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

API_URL = (
    "https://services9.arcgis.com/weJ1QsnbMYJlCHdG/arcgis/rest/services/"
    "Daily_Chokepoints_Data/FeatureServer/0/query"
)

PORTS = {
    "chokepoint4": "bab_el_mandeb",
    "chokepoint6": "hormuz",
}
START_DATE = "2019-01-01"
END_DATE = date.today().isoformat()  # до текущей даты
PAGE_SIZE = 1000

METRICS = [
    "n_container", "n_dry_bulk", "n_general_cargo", "n_roro", "n_tanker", "n_cargo", "n_total",
    "capacity_container", "capacity_dry_bulk", "capacity_general_cargo", "capacity_roro",
    "capacity_tanker", "capacity_cargo", "capacity",
]
FIELDS = ["date", "year", "month", "day", "portid", "portname", *METRICS]

# Папки, как в репозитории учебного проекта.
ROOT = Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": "academic-portwatch-notebook/1.0"})


def request_json(params, max_attempts=4):
    last_error = None
    for attempt in range(max_attempts):
        try:
            response = session.get(API_URL, params=params, timeout=90)
            response.raise_for_status()
            payload = response.json()
            if "error" in payload:
                raise RuntimeError(f"Ошибка API PortWatch: {payload['error']}")
            return payload
        except (requests.RequestException, ValueError) as exc:
            last_error = exc
            if attempt < max_attempts - 1:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Не удалось получить данные из PortWatch: {last_error}")


def fetch_port(portid):
    where = f"portid = '{portid}' AND year >= {pd.Timestamp(START_DATE).year}"
    expected = request_json({"where": where, "returnCountOnly": "true", "f": "json"}).get("count")
    if not isinstance(expected, int) or expected <= 0:
        raise RuntimeError(f"API не вернул количество записей для {portid}: {expected!r}")

    output = []
    offset = 0
    page_signatures = set()
    while offset < expected:
        params = {
            "where": where,
            "outFields": ",".join(FIELDS),
            "returnGeometry": "false",
            "orderByFields": "date ASC",
            "resultOffset": offset,
            "resultRecordCount": PAGE_SIZE,
            "f": "json",
        }
        features = request_json(params).get("features", [])
        if not features:
            raise RuntimeError(f"Преждевременный конец страниц {portid}: {offset} из {expected}")
        page = [item.get("attributes", {}) for item in features]
        signature = (str(page[0].get("date")), str(page[-1].get("date")), len(page))
        if signature in page_signatures:
            raise RuntimeError(f"API повторил страницу для {portid} на offset={offset}")
        page_signatures.add(signature)
        output.extend(page)
        offset += len(page)
        print(f"{PORTS[portid]}: получено {offset}/{expected} строк")

    if len(output) != expected:
        raise RuntimeError(f"Несовпадение числа строк {portid}: {len(output)} и {expected}")
    return output

all_rows = []
for portid in PORTS:
    print(f"\nЗагрузка {PORTS[portid]} ({portid})")
    all_rows.extend(fetch_port(portid))

if not all_rows:
    raise RuntimeError("PortWatch не вернул записей; пустые данные сохранять не будем")

df = pd.DataFrame(all_rows)
if pd.api.types.is_numeric_dtype(df["date"]):
    df["date"] = pd.to_datetime(df["date"], unit="ms", utc=True, errors="coerce")
else:
    df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
df["date"] = df["date"].dt.tz_localize(None).dt.normalize()

invalid_dates = int(df["date"].isna().sum())
if invalid_dates:
    raise ValueError(f"Обнаружены нераспознанные даты: {invalid_dates}")

df = df.loc[df["date"].between(pd.Timestamp(START_DATE), pd.Timestamp(END_DATE))].copy()
if df.empty:
    raise RuntimeError("В выбранном интервале не оказалось записей")

for metric in METRICS:
    if metric not in df.columns:
        raise RuntimeError(f"В ответе PortWatch нет поля {metric}")
    df[metric] = pd.to_numeric(df[metric], errors="coerce")

duplicates = int(df.duplicated(["date", "portid"]).sum())
df = df.sort_values(["portid", "date"]).drop_duplicates(["date", "portid"], keep="last")
df["route"] = df["portid"].map(PORTS)
df = df.reset_index(drop=True)

daily_cols = [c for c in ["date", "year", "month", "day", "portid", "portname", "route", *METRICS] if c in df.columns]
df = df[daily_cols]
print("\nГотово. Строк:", len(df), "| дубликатов убрано:", duplicates)
print("Фактический период:", df["date"].min().date(), "—", df["date"].max().date())
print(df.groupby("route")["date"].agg(["min", "max", "count"]).to_string())
display(df.head())

## 2. Сохранение ежедневных файлов (CSV / Excel / Parquet)

- `data/raw/portwatch_hormuz_bab_daily.csv` — стандартный CSV для Python и GitHub (разделитель `,`).
- `data/raw/portwatch_hormuz_bab_daily_excel.csv` — CSV с `;` и десятичной запятой для русскоязычного Excel.
- `data/raw/portwatch_hormuz_bab_daily.xlsx` — Excel с фильтром и зафиксированной шапкой.
- `data/raw/portwatch_hormuz_bab_daily.parquet` — для быстрой загрузки в Python.

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

csv_path = RAW_DIR / "portwatch_hormuz_bab_daily.csv"
csv_excel_path = RAW_DIR / "portwatch_hormuz_bab_daily_excel.csv"
xlsx_path = RAW_DIR / "portwatch_hormuz_bab_daily.xlsx"
parquet_path = RAW_DIR / "portwatch_hormuz_bab_daily.parquet"

df.to_csv(csv_path, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
df.to_csv(csv_excel_path, index=False, sep=";", decimal=",", encoding="utf-8-sig", date_format="%Y-%m-%d")
df.to_parquet(parquet_path, index=False)

with pd.ExcelWriter(xlsx_path, engine="openpyxl", datetime_format="yyyy-mm-dd") as writer:
    df.to_excel(writer, index=False, sheet_name="PortWatch daily")
    ws = writer.sheets["PortWatch daily"]
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="1F4E78")
        cell.alignment = Alignment(wrap_text=True)
    for i, col_cells in enumerate(ws.columns, 1):
        sample = list(col_cells)[:150]
        width = min(max(max((len(str(c.value or "")) for c in sample), default=8) + 2, 12), 25)
        ws.column_dimensions[get_column_letter(i)].width = width
    for cell in ws["A"][1:]:
        cell.number_format = "yyyy-mm-dd"

for p in (csv_path, csv_excel_path, xlsx_path, parquet_path):
    print("Сохранено:", p)

## 3. Помесячная таблица для команды

Для совместного EDA требуется период **2015-04-01 — 2026-09-01**. PortWatch начинается в 2019 году, поэтому месяцы до этого будут **пустыми, а не нулевыми**. Суточные числовые показатели усредняем внутри каждого месяца и маршрута; дополнительно сохраняем количество наблюдаемых дней и долю покрытия месяца.

**Внимание:** среднее за месяц — не фактический месячный объём перевезённого карбамида. Для моделирования не подбирайте параметры на отложенной тестовой выборке.

In [ ]:
MONTH_START = "2015-04-01"
MONTH_END = "2026-09-01"  # требование чекпоинта 2

monthly_long = df.copy()
monthly_long["date"] = monthly_long["date"].dt.to_period("M").dt.to_timestamp()

agg = monthly_long.groupby(["date", "route"])[METRICS].mean().reset_index()
observed = monthly_long.groupby(["date", "route"])["date"].size().rename("days_observed").reset_index()
agg = agg.merge(observed, on=["date", "route"], how="left")
agg["days_in_month"] = agg["date"].dt.days_in_month
agg["coverage_share"] = agg["days_observed"] / agg["days_in_month"]

pieces = []
for route in PORTS.values():
    x = agg.loc[agg["route"] == route].drop(columns="route").copy()
    x = x.rename(columns={metric: f"{route}_{metric}_mean" for metric in METRICS})
    x = x.rename(columns={m: f"{route}_{m}" for m in ["days_observed", "days_in_month", "coverage_share"]})
    pieces.append(x)

month_index = pd.DataFrame({"date": pd.date_range(MONTH_START, MONTH_END, freq="MS")})
monthly = month_index.copy()
for x in pieces:
    monthly = monthly.merge(x, on="date", how="left")

monthly_path = PROCESSED_DIR / "portwatch_hormuz_bab_monthly.csv"
monthly_parquet = PROCESSED_DIR / "portwatch_hormuz_bab_monthly.parquet"
monthly.to_csv(monthly_path, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
monthly.to_parquet(monthly_parquet, index=False)
print("Сохранено:", monthly_path, "| строк:", len(monthly))
print("Сохранено:", monthly_parquet)
print("Месячные даты:", monthly["date"].min().date(), "—", monthly["date"].max().date())
display(monthly.loc[monthly["date"].between("2023-01-01", "2023-05-01")].head())

## 4. Быстрая проверка и графики

Покажем среднемесячное число **сухогрузов** и оценочную суммарную грузовместимость сухогрузов по двум проливам. Для детального EDA (пропуски, выбросы, ошибки, тренды) используйте отдельный отчёт проекта.

In [ ]:
for metric, title, ylabel in [
    ("n_dry_bulk", "Среднее число сухогрузов в день (по месяцам)", "Судов/день"),
    ("capacity_dry_bulk", "Средняя оценочная вместимость сухогрузов за день (по месяцам)", "Оценочная вместимость, тонн"),
]:
    fig, ax = plt.subplots(figsize=(12, 4))
    for route in PORTS.values():
        x = monthly[["date", f"{route}_{metric}_mean"]].dropna()
        ax.plot(x["date"], x[f"{route}_{metric}_mean"], label=route.replace("_", " "))
    ax.set_title(title)
    ax.set_xlabel("Дата")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

print("\nДней на пролив (проверка полноты в суточных данных):")
for route, group in df.groupby("route"):
    period = pd.date_range(group["date"].min(), group["date"].max(), freq="D")
    missing = period.difference(group["date"])
    print(f"{route}: наблюдений {len(group)}, отсутствующих дат внутри периода {len(missing)}")

print("\nПустые числовые значения:")
display(df.groupby("route")[METRICS].apply(lambda x: x.isna().sum()))

## 5. Если работаешь в Google Colab — скачать файлы на компьютер

В обычном Jupyter файлы уже сохранены в папках `data/raw` и `data/processed`; автоматически скачивать их не нужно.

In [ ]:
try:
    from google.colab import files
    for p in (csv_path, xlsx_path, parquet_path, monthly_path):
        files.download(str(p))
except ImportError:
    print("Обычный Jupyter: файлы лежат в data/raw и data/processed")

## Замечания для отчёта и команды

1. `n_dry_bulk` — проходы сухогрузов; `capacity_dry_bulk` — их оценочная грузовместимость. Ни одна из этих величин не равна массе перевезённого карбамида.
2. По историческим данным **до 2019 года** нет покрытия PortWatch; нельзя заполнять эти месяцы нулями.
3. Последний месяц может быть неполным. Для проверки есть `days_observed` и `coverage_share`.
4. Для ценовых рядов объединяйте таблицы по колонке `date` (первое число месяца).
5. Если используете временные разбиения train/validation/test, определите их до обработки данных, которая обучается на распределении (импутация, масштабирование и т.п.). Не анализируйте тест для подбора модели.